In [3]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
!pip install holidays
import holidays

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Processo Seletivo Analytica 2026 - **Grupo L**

# Requisitos
- Prever a duração das viagens de ônibus, com base nas viagens antigas
- Calcular o tempo de ciclo de uma linha de ônibus dado um horário (duraçao ida + duraçao volta
- Dado o tempo de ciclo, calcular quantos veículos devem estar em uma linha dado um horário, para que o intervalo de saída de ônibus informado pela prefeitura seja respeitado
- EXTRA: Cruzar dados das “piores linhas” com os bairros, buscando os bairros mais afetados
- EXTRA: Adicionar dados no modelo preditivo: mm de chuva nas últimas 2/3 horas e o esperado para próxima hora quando o ônibus sair do terminal (fontes: https://datariov2-pcrj.hub.arcgis.com/datasets/25eceb6bd2214fbb8514e9cdf8e1f0f0_1/explore


# Preparação de dados

Para esse problema, realizamos uma busca dentre diversos datasets e encontramos os seguintes dados disponíveis. Abaixo, mostrarei uma visualização de algumas tabelas de dados.

**Fonte extra de dados**: https://www.data.rio/documents/transporte-rodovi%C3%A1rio-viagens-dos-%C3%B4nibus-identificadas-por-gps/about e https://github.com/prefeitura-rio/queries-rj-smtr/tree/master/models/projeto_subsidio_sppo

# **OBS**:
Para visualizar outras tabelas csv nos códigos abaixo, basta substituir a string dentro de `caminho_dado` pelo caminho do arquivo que se deseja visualizar. Todos os arquivos utilizados estão na pasta `/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL`. Assim que tiver o e-mail de vocês, eu compartilho ela no drive.

## SMTR — Sistema Municipal de Transportes

Dados operacionais, financeiros e estruturais do sistema de transporte público do Rio de Janeiro. Filtrei alguns dados e, abaixo, segue a visualização de alguns que se mostraram mais relevantes para atender os requisitos.

### dashboard_bilhetagem_implantacao_jae
Monitoramento GPS Descrição: Dados agregados de localização de veículos por modal:

- BRT
- Ônibus (SPPO)
- Vans (STPL)
- VLT




In [ ]:

caminho_dado = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/dashboard_bilhetagem_implantacao_jae-20260430T211213Z-3-001/dashboard_bilhetagem_implantacao_jae/gps_agregado_onibus.csv"
df = pd.read_csv(caminho_dado)
display(df.iloc[1000:1250])
print((df['estado_equipamento'] == 'FECHADO').sum())
print(df.shape)
df.info()
df.describe()


### gtfs
Base padronizada de transporte contendo:
- routes: linhas
- stops: paradas(id das paradas e localização). 485884 linhas
- trips: viagens (onibus, id das rotas e direção) 556360 linhas
- shapes: geometria das rotas (ponto de inicio/fim e formato). 13946 linhas. **aponta para shape**
- calendar + calendar_dates: operação temporal (obras, dias da semana, dia de inicio e término). 923 linhas e 34225 linhas
- ordem_servico: onibus, horarios de funcionamento, tamanho do trajedo ida/volta, total viagens ida/volta (dado o dia da semana) 113629 linhas
- frequencies: contém o intervalo entre um onibus e outro, delimitado pela prefeitura




In [4]:

caminho_dado = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/gtfs-20260430T211243Z-3-001/gtfs/frequencies.csv"
df = pd.read_csv(caminho_dado)
display(df.iloc[1:120])
print(df.shape)
df.info()
df.describe()


,feed_version,feed_start_date,feed_end_date,trip_id,start_time,end_time,headway_secs,exact_times,versao_modelo
1,2023-06-01,2023-06-01,2023-06-15,55f4e628-7e93-4183-b766-de2524df1c3b,17:11:00,20:23:00,1440,0,201d79faee763526a030ff998bebea9782efe961
2,2026-02-02,2026-02-02,2026-02-10,ca93c752-a3d8-4ba5-b6b9-a4a92385726c,05:00:00,06:00:00,360,0,f7495a359559836032529b2d93bc4eff1a079265
3,2024-05-15,2024-05-15,2024-06-02,3f6c3eae-c4d8-444c-9838-d0429a28ffd4,07:46:00,08:10:00,720,0,31a6853cb59b7bd67d2f865fa81c6992ac4e1435
4,2025-01-25,2025-01-25,2025-02-09,0310c444-502c-4729-aba9-d055158e9852,17:45:00,18:11:00,1560,0,6f4042d043fc67e3b7822b43f11afe7aab4b22d3
5,2023-12-16,2023-12-16,2023-12-20,42a51dbd-7f02-43f4-8475-bdb0567468ae,13:03:30,13:47:00,435,0,201d79faee763526a030ff998bebea9782efe961
...,...,...,...,...,...,...,...,...,...
115,2024-06-03,2024-06-03,2024-06-04,99d85c09-71e9-40af-9dc6-f326dc118f2c,22:28:00,26:16:00,1140,0,31a6853cb59b7bd67d2f865fa81c6992ac4e1435
116,2026-01-03,2026-01-03,2026-01-18,816a2d43-3606-4720-bca7-f65ec671edc3,06:03:00,06:57:00,1620,0,8df7635a5781e2e43ecc1920667b18f29516caa0
117,2024-12-31,2024-12-31,2025-01-01,a56d1407-71b1-4a99-ba8c-e038b88789a2,15:13:00,15:35:00,1320,0,3e1630ddb15dca783e340faf23f08f68733f4813
118,2023-03-16,2023-03-16,2023-03-31,37edb768-a5a1-44b3-9dc3-b2c9785be9cd,20:51:00,24:27:00,4320,0,201d79faee763526a030ff998bebea9782efe961


(484555, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 484555 entries, 0 to 484554
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   feed_version     484555 non-null  object
 1   feed_start_date  484555 non-null  object
 2   feed_end_date    479100 non-null  object
 3   trip_id          484555 non-null  object
 4   start_time       484555 non-null  object
 5   end_time         484555 non-null  object
 6   headway_secs     484555 non-null  int64 
 7   exact_times      484555 non-null  int64 
 8   versao_modelo    484555 non-null  object
dtypes: int64(2), object(7)
memory usage: 33.3+ MB


,headway_secs,exact_times
count,484555.000000,484555.0
mean,1463.111506,0.0
std,1104.580663,0.0
min,60.000000,0.0
25%,600.000000,0.0
50%,1200.000000,0.0
75%,1800.000000,0.0
max,10800.000000,0.0


### br_rj_riodejaneiro_viagem_zirix
**TABELA CENTRAL**

Dados
- Horas de inicio e chegada de cada linha
- **Aponta para o shape id, veículo e número da linha**

Útil para calcular a duração de cada viagem dada a hora, que pode servir para criar um modelo preditivo que alimenta possíveis cálculos de

```
# Isto está formatado como código
```

otimizações.





In [ ]:

caminho_dado = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/br_rj_riodejaneiro_viagem_zirix-20260430T211251Z-3-001/br_rj_riodejaneiro_viagem_zirix/viagem_informada.csv"
df = pd.read_csv(caminho_dado)
display(df.iloc[250000:251000])
print(df.shape)
df.info()
df.describe()


### veiculo
Dados dos veículos do sistema:

- Licenciamento: id de veículo, data da última vistoria, tecnologia, id chassi. 222625 linhas.
- solicitação de licenciamento: solicitações realizadas. Acredito que não está no escopo
- registro agente verao: Acredito que não está no escopo
# - Operação diária
- Inspeções
- Condições operacionais

In [ ]:
"""
caminho_dado = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/veiculo-20260430T211305Z-3-001/veiculo/sppo_veiculo_dia.csv"
df = pd.read_csv(caminho_dado)
display(df.iloc[1:120])
print(df.shape)
df.info()
df.describe()
"""

# Solução proposta

Com base nos dados acima, iremos analisar os dados para aplicar otimizações na quantidade de veículos por linha de ônibus. O objetivo é diminuir o tempo de espera dos passageiros por linha de ônibus.

Utilizaremos o o intervalo `headway_secs` de início de linha delimitado pela prefeitura na tabela `frequencies.csv` do `gtfs` e, com ele, definiremos que o intervalo médio de espera em um ponto é `headway_secs`/2

(Necessário embasar essa conta acima, sei que ela vem de probabilidade se a gente considerar que ass pessoas chegam nos pontos de forma uniforme e aleatória.)

Em adição, criaremos um modelo preditivo que receberá o tempo de viagem descrito na tabela `viagem_informada.csv` do dataset `br_rj_riodejaneiro_viagem_zirix`, em adição da linha, sentido, horário e dia da semana. O modelo deverá ser  capaz de prever a duração da viagem em qualquer horário e dia futuro.

(Ainda não sei se uma regressão polinomial para cada uma das linhas vai servir para descrever esse tempo de deslocamento dado a hora e dia)

Uniremos os dados de tempo de viagem e intervalo médio nos pontos e, com isso, calcularemos a quantidade de ônibus necessários que essa linha deve possuir naquele momento para cumprir o intervalo médio.

Podemos realizar uma análise dos dados fornecidos e indicar o atraso médio por linha e informações similares, demonstrando a necessidade da nossa aplicação.

Se sobrar tempo, podemos melhorar o modelo para realocar ônibus entre as linhas de forma dinâmica durante o dia, para servir de apoio a secretaria de transportes e empresas de ônibus.

(falta adicionar um tempo de repouso na conta)
(falta embasar esse problema - má distribuição de frota de onibus rj)

## Limitações
Com os dados de GPS, seria possível otimizar o modelo para estimar o deslocamento entre trechos da rota com todas as linhas que cruzam esses trechos, sem se limitar a uma análise que considera apenas os ônibus da mesma linha.





#Dados necessários para o modelo preditivo

1. frequencies.csv (GTFS)
	1. start_time e end_time. Dados sobre hora do dia que a regra abaixo deve valer
	2. headway_secs . Tempo que um ônibus leva para chegar no mesmo ponto que um ônibus acabou de passar (usado para conta de ônibus em linha)
	3. trip_id. Id da viagem. Identifica o dado com a viagem
2. trips.csv(GTFS)
	1. trips_id. Utilizado para dar merge com a tabela acima
	2. trip_short_name: Nome da linha (ex: 324)
	3. direction_id: Direção ida ou volta.
3. ordem_servico_csv (GTFS)
	1. servico (cruzar com viagens_informadas
	2. tamanho da rota ida/tamanho da rota volta
4. calendar.csv (GTFS)
4. viagem_informada.csv (br_rj_riodejaneiro_viagem_zirix-20260430T211251Z-3-001/br_rj_riodejaneiro_viagem_zirix)
	1. servico: será usada para cruzar com trip_short)name da tabela acima
	2. sentido: Teremos que levar em consideração a Ida e Volta da linha, para cruzar com 0 ou 1 do direction_id da tabela acima. No caso, Ida = 0 e Volta = 1
	3. datetime_partida e datetime_chegada: Usados para calcular o tempo de duração de uma viagem de ônibus

O objetivo é juntar todos os dados acima em uma única tabela, para facilitar a manipulação de dados. Porém, antes de uni-las, é importante tratar valores nulos, inválidos e outliers. Segue abaixo o algoritmo que busca por essas situações.

# Limpeza básica de dados com análise exploratória simples

In [ ]:


caminho_frequencies = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/gtfs-20260430T211243Z-3-001/gtfs/frequencies.csv"
caminho_trips = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/gtfs-20260430T211243Z-3-001/gtfs/trips.csv"
caminho_viagem = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/br_rj_riodejaneiro_viagem_zirix-20260430T211251Z-3-001/br_rj_riodejaneiro_viagem_zirix/viagem_informada.csv"

caminhos = [caminho_frequencies, caminho_trips, caminho_viagem]

for caminho in caminhos:
  df = pd.read_csv(caminho)
  display(df.isnull().sum())
  df.info()

Existem 9 linhas em trips.csv que possuem valores nulos para o sentido da viagem. Abaixo veremos quais linhas são essas:

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/gtfs-20260430T211243Z-3-001/gtfs/trips.csv")
display(df[df['direction_id'].isnull()])

São linhas alternativas, algumas para eventos como reveillon. Decidi excluir do dataset

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/gtfs-20260430T211243Z-3-001/gtfs/trips.csv")
df_trips_limpo = df.dropna(subset=['direction_id']).reset_index(drop=True) #esse reset index é para atualizar o indice apos a exclusao
caminho_trips_limpo = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados tratados/trips_limpo.csv"
df_trips_limpo.to_csv(caminho_trips_limpo, index=False)
df_trips_limpo.isnull().sum()


Verificarei abaixo se existem outliers na coluna de `headway_secs` de `frequencies.csv`

In [ ]:
#necessário converter o headway de string para int primeiro
df = pd.read_csv("/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/gtfs-20260430T211243Z-3-001/gtfs/frequencies.csv")
df_freq_limpo = df
df_freq_limpo['headway_secs'] = df['headway_secs'].astype(int)
caminho_freq_limpo = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados tratados/freq_limpo.csv"
df_freq_limpo.to_csv(caminho_freq_limpo, index=False)

df.plot(kind='box', figsize=(30, 5), subplots=True, layout=(1, 8))
plt.figure()



Temos poucos valores distantes próximos a 10.000 segundos. Isso equivale a aproximadamente 3 horas. Provavelmente são ônibus que funcionam de madrugada ou que possuem poucos horários por dia. Não temos outliers que necessitem ser excluídos.



# Cruzamento de trips, frequencies e calendar

1.   Item da lista
2.   Item da lista

headway_secs

Para obter o dimensionamento da frota de ônibus, vamos precisar dos headway_secs determinados pela prefeitura para cada horário e dia da semana. Para isso, precisamos cruzar as tabelas `trips.csv`, `frequencies.csv` e `calendar.csv`.

Uma das dificuldades é que o intervalo da duração da regra do GTFS está em um formato que permite mais de 24 horas, sendo o horário depois de 24 equivalente a madrugada do dis seguinte. Precisamos converter para minutos depois da meia noite, para se tornar compatível com os dados do modelo.Segue abaixo o código que faz isso

In [ ]:
def converter_horario_gtfs(horario_str):
    #separa a string e calcula minutos desdde meia noite
    partes = str(horario_str).split(':')
    horas = int(partes[0])
    minutos = int(partes[1])
    minutos_totais = (horas * 60) + minutos

    #aplica modulo para aplicar o comportamento do relogio
    minutos_reais = minutos_totais % 1440

    return minutos_reais

# Realizando o cruzamento (Join)
df_trips = pd.read_csv("/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados tratados/trips_limpo.csv").copy()
df_frequencies = pd.read_csv("/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados tratados/freq_limpo.csv").copy()

#cruzando coluna trips e frequencies
df_headways_por_linha = pd.merge(
    df_trips,
    df_frequencies,
    how='inner' # 'inner' garante que só vamos manter as viagens que têm headway definido
)

#cruzando coluna resultante com calendar
df_calendar = pd.read_csv("/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/gtfs-20260430T211243Z-3-001/gtfs/calendar.csv").copy()
df_headways_por_linha = pd.merge(df_headways_por_linha, df_calendar, on=['feed_version', 'feed_start_date', 'feed_end_date', 'versao_modelo', 'service_id'], how='inner')

#excluindo colunas inuteis
lista_dropar = [
    'feed_version', 'feed_start_date', 'feed_end_date', 'versao_modelo',
    'route_id', 'trip_headsign', 'block_id',
    'shape_id', 'wheelchair_accessible', 'bikes_allowed', 'exact_times'
]
df_headways_por_linha.drop(columns=lista_dropar, inplace=True, errors='ignore')

# Removendo qualquer duplicata que ainda possa ter sobrado por sujeira nos dados brutos
df_headways_por_linha.drop_duplicates(inplace=True)

# Convertendo o intervalo de duração da regra do headway_secs para minutos desde meia noite
df_headways_por_linha['start_time'] = df_headways_por_linha['start_time'].apply(converter_horario_gtfs)
df_headways_por_linha['end_time'] = df_headways_por_linha['end_time'].apply(converter_horario_gtfs)

#retirando o itinerario de obras e situaçoes anormais
# Lista de calendários de operação normal (sem obras ou megaeventos)
servicos_validos = ['U_REG', 'S_REG', 'D_REG', 'U', 'S', 'D']

# Filtra a base mantendo APENAS a operação padrão
df_headways_por_linha = df_headways_por_linha[df_headways_por_linha['service_id'].isin(servicos_validos)]

#salva tabela no drive
df_headways_por_linha.to_csv("/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados tratados/headways_por_linhas.csv", index=False)

display(df_headways_por_linha.iloc[100:1000])
display(df_headways_por_linha.info())





# Adição da duração da viagem no dataset
Abaixo, vou incluir a coluna `duracao_viagem` na tabela `viagem_informada.csv`. Em seguida, realizarei a análise buscando por outliers (viagens excessivamente longas ou curtas demais).

In [ ]:
#criando cópia na memória
df = pd.read_csv("/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/br_rj_riodejaneiro_viagem_zirix-20260430T211251Z-3-001/br_rj_riodejaneiro_viagem_zirix/viagem_informada.csv")
df_viagens_limpo = df.copy()

#convertendo os valores de data, hora, minuto e segundo de string para datetime64
df_viagens_limpo['datetime_chegada'] = pd.to_datetime(df_viagens_limpo['datetime_chegada'])
df_viagens_limpo['datetime_partida'] = pd.to_datetime(df_viagens_limpo['datetime_partida'])

#incluindo a coluna duracao_viagem_minutos nesse dataset
df_viagens_limpo['viagem_minutos'] = (df_viagens_limpo['datetime_chegada'] - df_viagens_limpo['datetime_partida']).dt.total_seconds() / 60
display(df_viagens_limpo.head())



## Quantidade de viagens por linha
Será que essa distribuição é desigual Temos linhas com poucas entradas de viagens?

O código abaixo identifica as linhas que possuem menos do que 200 viagens registradas e exclui-as do dataset. Perdemos apenas 5% das entradas, que estavam associadas a aproximadamente 40% das linhas.

O modelo vai perder a capacidade de realizar previsões com essas linhas, porém, com o dataset atual essa medida foi necessária para evitar overfitting

**Altere o limite corte para verificar quantas linhas possuem menos ou mais do que esse número de viagens**

In [ ]:
# calculando a contagem de viagens p cada linha
contagem_viagens = df_viagens_limpo['servico'].value_counts()
lista_linhas = df_viagens_limpo['servico'].unique()

print("Total de linhas:" + str(len(lista_linhas)))

# criando o histograma dessa contagem
plt.figure(figsize=(12, 6))
plt.hist(contagem_viagens, bins=50, color='salmon', edgecolor='black')

# Adicionando uma linha vertical para facilitar a identificaç~ao da quantidade de linhas com menos de x viagens
limite_corte = 200
plt.axvline(limite_corte, color='red', linestyle='--', label='Sugestão de Corte ('+str(limite_corte)+' viagens)')
print("Linhas com menos de "+str(limite_corte)+" viagens: "+str( (contagem_viagens < limite_corte).sum() ))
print("Linhas com mais ou igual de "+str(limite_corte)+" viagens: "+str( (contagem_viagens >= limite_corte).sum() ))

plt.title('Distribuição da Quantidade de Viagens por Linha')
plt.xlabel('Número de Viagens que a linha possui')
plt.ylabel('Quantidade de Linhas')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

# DELETANDO AS VIAGENS DAS LINHAS QUE POSSUEM MENOS DE "LIMITE_CORTE" VIAGENS
linhas_aprovadas = contagem_viagens[contagem_viagens >= 200].index
df_viagens_limpo = df_viagens_limpo[df_viagens_limpo['servico'].isin(linhas_aprovadas)].copy()

In [ ]:
# Quanto do volume total (em %) essas linhas com poucos dados de viagens representam?
viagens_pequenas = contagem_viagens[contagem_viagens < limite_corte].sum()
total_geral = contagem_viagens.sum()

print(f"Total de viagens: {total_geral} viagens")
print(f"Quantidade de viagens de linhas pequenas: {viagens_pequenas} viagens")
print(f"Porcentagem equivalente do total de dados: {(viagens_pequenas / total_geral) * 100:.2f}%")

# Análise Exploratória de Dados Inicial

## Duração das viagens
 É possível ver que existem viagens com duração menor que 10 minutos sendo a menor viagem com duração negativa e a maior com mais de 12 horas (outlier). Temos que excluir essas viagens "extremas" do dataset

In [ ]:

#buscando por outliers
df_viagens_limpo['viagem_minutos'].hist(figsize=(30, 5), range=(0,15))
plt.figure()
df_viagens_limpo['viagem_minutos'].hist(figsize=(30, 5), range=(0,30))
plt.figure()
df_viagens_limpo['viagem_minutos'].hist(figsize=(30, 5), range=(0,120))
plt.figure()

#técnica proposta pela monitora na aula de tratamento de dados. Vai dar os valores minimos, médios e máximos das durações da viagem
print(df_viagens_limpo[['servico', 'viagem_minutos']].describe().apply(lambda s: s.apply('{0:.2f}'.format)))




In [ ]:
# Agrupamos por serviço e descrevemos a coluna de minutos
resumo_por_linha = df_viagens_limpo.groupby('servico')['viagem_minutos'].describe()

# Formatamos para 2 casas decimais para facilitar a leitura
print(resumo_por_linha.apply(lambda s: s.apply('{0:.2f}'.format)))

## Identificando outliers: viagens muito rápidas ou muito lentas.

Para identificar esses outliers, utilizarei a função `outlier`, definida abaixo. Uma função semelhante foi apresentada na aula de Tratamento de Dados da Analytica, mas optei por essa pois os dados de trânsito não são simétricos. Utilizar IQR não foi eficiente.

Para o nosso contexto, precisamos agrupar o número de viagens por linhas de ônibus.


Inicialmente,excluí todas as viagens mais curtas que 10 minutos e tentei filtrar usando intervalos interquartis e medianas, porém, isso acabou permitindo viagens muito rápidas/curtas ou então excluindo viagens rápidas.

Como melhoria, decidi mudar de método e cruzar a duração do percurso com a distância percorrida, fixando uma velocidade média mínima (5km/h) e máxima(65km/h) e, assim, filtrar os ônibus que não possuírem uma velocidade média dentro do intervalo estipulado. Esses serão tratados como outliers, representando medições erradas ou situações fora do comum nas vias.(https://www.rio.rj.gov.br/web/guest/exibeconteudo?id=1893686 ).

In [ ]:
def outlier(df, coluna):

  p10 = df[coluna].quantile(0.10)
  p90 = df[coluna].quantile(0.90)
  mediana = df[coluna].median()

  #30% da mediana é tolerável. Se a mediana for 20km/h, aceita até 6km/h.
  limite_inferior = mediana * 0.3

  # Toda a linha tem tolerância para chegar a pelo menos 50 km/h (madrugadas/vias livres).
  # Se for um BRT rápido (onde P90*1.5 dá mais de 50), ele aceita o maior valor, travado no máximo físico de 65 km/h.
  limite_superior = min(max(p90 * 1.8, 50.0), 65.0)

  df_bom = df[(df[coluna] >= limite_inferior) & (df[coluna] <= limite_superior) | df[coluna].isna()].copy()

  # Teste 1: Velocidade media

  # Isola quem são os outliers para podermos imprimir os detalhes
  df_ruim = df[(df[coluna] < limite_inferior) | (df[coluna] > limite_superior)]

  if len(df_ruim) > 0:
      print(f"\n{len(df_ruim)} OUTLIER(S) ENCONTRADO(S) NA LINHA '{df['servico'].iloc[0]}'. Viagens válidas: '{len(df_bom)}'")

      # Faz um loop pelos outliers para mostrar o valor e o intervalo
      cont = 0
      for index, row in df_ruim.iterrows():
          valor_exato = row[coluna]
          print(f"   -> Viagem de {valor_exato:.2f} hm/h | (Intervalo aceitável era de {limite_inferior:.2f} a {limite_superior:.2f})")
          cont += 1
          if(cont >5):
            cont = 0
            break

  return df_bom

def limpar_outlier_duracao_viagem(df):
  df_limpo = []
  linhas_onibus = df['servico'].unique()

  for linha in linhas_onibus:
    df_linha = df[df['servico'] == linha].copy()
    df_sem_outlier = outlier(df_linha, 'veloc_media_kmh')
    df_limpo.append(df_sem_outlier)

  df_final = pd.concat(df_limpo, ignore_index=True)
  return df_final

cont_removidas = 0
cont_total = len(df_viagens_limpo)

df_ordem_servico = pd.read_csv("/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados brutos/gtfs-20260430T211243Z-3-001/gtfs/ordem_servico.csv")
#remove duplicatas e filtra a parte util
df_ordem_servico = df_ordem_servico[['servico', 'extensao_ida', 'extensao_volta']].drop_duplicates(subset=['servico'])

#cruza os dados
df_viagens_limpo = df_viagens_limpo.merge(df_ordem_servico, on='servico', how='left')

# Definimos a distância correta da viagem baseada na coluna 'sentido' (0 = Ida, 1 = Volta)
df_viagens_limpo['distancia_km'] = np.where(
    df_viagens_limpo['sentido'] == 0,
    df_viagens_limpo['extensao_ida'],
    df_viagens_limpo['extensao_volta']
)

# Calculamos a velocidade média em km/h
df_viagens_limpo['veloc_media_kmh'] = df_viagens_limpo['distancia_km'] / (df_viagens_limpo['viagem_minutos'] / 60)

# remove viagens sem valor de velocidade
cont_removidas = df_viagens_limpo['veloc_media_kmh'].isna().sum()
print("Viagens com velocidade nula - " + str(cont_removidas))
df_viagens_limpo = df_viagens_limpo.dropna(subset=['veloc_media_kmh']).copy()

#excluindo casos extremos de velocidade
vel_minima = 5.0
vel_maxima = 65.0
df_valido = (df_viagens_limpo['veloc_media_kmh'] >= vel_minima) & (df_viagens_limpo['veloc_media_kmh'] <= vel_maxima)



# Atualiza o DataFrame APENAS com os dados fisicamente possíveis
df_viagens_limpo = df_viagens_limpo[df_valido].copy()

print(f"Viagens com velocidade absurda (fora da faixa 5 a 65 km/h): {cont_total - cont_removidas - len(df_viagens_limpo)}")
cont_removidas += cont_total - cont_removidas - len(df_viagens_limpo)

df_viagens_limpo = limpar_outlier_duracao_viagem(df_viagens_limpo)






In [ ]:
display(df_viagens_limpo.head())

# Limpeza final da tabela: Conversão dos valores para tipos adequados para o modelo de ML baseado em árvores

Segundo [blog técnico da NVIDIA](https://developer.nvidia.com/blog/three-approaches-to-encoding-time-information-as-features-for-ml-models/), "when using non-linear models such as decision trees (or ensembles of thereof), we do not explicitly encode features such as month number or day of the year as dummies. Those models are capable of learning non-monotonic relationships between ordinal input features and the target".

Logo, não precisaremos utilizar técnicas de OHC e dummy variables

Assim, uma ideia para as colunas que alimentarão o modelo pode ser:
1. `mês_viagem`:  valor inteiro de 1 a 12
1. `dia_viagem`: valor numerico (1 a 30) do dia da viagem.
2. `dia_semana`:  valor inteiro de 1 a 7 (segunda a domingo)
1. `eh_feriado`: 0 se for feriado e 1 se nao
2. `hora_partida`:  valor inteiro representando a quantidade de minutos que se passaram desde meia noite (0 ate 1429)
2. `duracao`:  valor inteiro representando a quantidade de minutos que se passaram desde meia noite (0 ate 1429) **VALOR QUE SERA PREVISTO**
3. `linha`: numero representando a linha. sera necessário fazer um hashmap entre a string da linha (ex. "SN324) e um numero inteiro representando-a
3. `sentido`: Indica se a viagem estava indo (0) ou voltando (1)

Abaixo, temos o algoritmo que estrutura todos esses valores em uma única tabela:




In [ ]:
df_final = df_viagens_limpo.copy()

# excluindo colunas desnecessarias
lista_dropar = ['data', 'datetime_chegada', 'datetime_processamento', 'datetime_captura', 'id_veiculo', 'trip_id', 'route_id', 'shape_id', 'id_viagem', 'versao', 'datetime_ultima_atualizacao', 'extensao_ida', 'extensao_volta', 'distancia_km', 'veloc_media_kmh']
df_final = df_final.drop(lista_dropar, axis=1)

df_final = df_final.rename(columns={
    'viagem_minutos': 'duracao'
})

#Desmembrar a coluna de datetime_partida em mes, dia, hora, dia da semana e feriado

df_final['mes_viagem'] = df_final['datetime_partida'].dt.month
df_final['dia_viagem'] = df_final['datetime_partida'].dt.day
df_final['hora_partida'] = (df_final['datetime_partida'].dt.hour * 60) + df_final['datetime_partida'].dt.minute
#Dia da Semana (0 = Segunda-feira, 6 = Domingo)
df_final['dia_semana'] = df_final['datetime_partida'].dt.dayofweek

#adicionando feriados
data_feriados = holidays.country_holidays('BR', subdiv='RJ')
df_final['eh_feriado'] = df_final['datetime_partida'].dt.date.isin(data_feriados).astype(int)

df_final = df_final.drop('datetime_partida', axis=1)

display(df_final.iloc[100:1000])

#Convertendo string Ida e VOlta de sentido para 0 e 1
df_final['sentido'] = df_final['sentido'].map({'Ida': 0, 'Volta': 1})


#salvando dataframe
caminho_df_final = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados tratados/df_final.csv"
df_final.to_csv(caminho_df_final, index=False)


In [ ]:
df_final['duracao'].hist(bins=50)

plt.show()

In [ ]:
df_join = df_final.merge(df_ordem_servico, on='servico', how='left')
df_join[df_join.duracao > 200]

# Gerando alguns gráficos


In [ ]:

df_plot = df_viagens_limpo.copy()

# extrair hora do dia (0 a 23) e o dia da semana (0=Seg, 6=Dom) diretamente
df_plot['hora_cheia'] = df_plot['datetime_partida'].dt.hour
df_plot['dia_semana'] = df_plot['datetime_partida'].dt.dayofweek

# filtrar apenas os dias úteis
df_dias_uteis = df_plot[df_plot['dia_semana'] <= 4]

veloc_media_por_hora = df_dias_uteis.groupby('hora_cheia')['veloc_media_kmh'].mean()

# gerar o gráfico
ax = veloc_media_por_hora.plot(
    kind='bar',
    figsize=(12, 6),
    color='#ff7f0e',
    edgecolor='black'
)

ax.set_title('Velocidade Média dos Ônibus por Hora (Apenas Dias Úteis)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hora do Dia (0h - 23h)', fontsize=12)
ax.set_ylabel('Velocidade Média (km/h)', fontsize=12)

plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Exibir o gráfico
plt.show()

In [ ]:
df_dias_uteis = df_final[df_final['dia_semana'] <= 4].copy()
df_dias_uteis['hora_cheia'] = df_dias_uteis['hora_partida'] // 60

#agrupar pela hora cheia e calcular a média da coluna 'duracao'
duracao_media_por_hora = df_dias_uteis.groupby('hora_cheia')['duracao'].mean()

#gerar o gráfico
ax = duracao_media_por_hora.plot(
    kind='bar',
    figsize=(12, 6),
    color='#1f77b4',
    edgecolor='black'
)

ax.set_title('Duração Média das Viagens por Hora (Apenas Dias Úteis)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hora do Dia (0h - 23h)', fontsize=12)
ax.set_ylabel('Duração Média (Minutos)', fontsize=12)

# Melhorando a leitura
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
## exclusão de dados de viagens com durações descrepantes
df_final_limpo = df_final[df_final.duracao < 200].copy()

caminho_df_final_limpo = "/content/drive/MyDrive/DadosAnalyticaPS2026-GrupoL/dados tratados/df_final_limpo.csv"
df_final_limpo.to_csv(caminho_df_final_limpo, index=False)